In [ ]:
BANNER = \
"""# ----------------------------------------------------------------------
# PTS | Paulo Trigo Silva
# mLDm | MoP
# v01
# ----------------------------------------------------------------------
"""

In [ ]:
# ------------------------------------------------------------
# from model selection to deployment
# ------------------------------------------------------------

# MoP_04 | From Dataset to Model
# notebook 3 | model selection and deployment

# input:
#   03_output/penguins_modelling_[A-E]_*.csv
#   03_output/penguins_modelling_[A-E]_*.joblib
#   NOTE:
#   - datasets and feature transformers were generated in notebook 1
#   - model selection is informed by the experiments in notebook 2
#
# output:
#   03_output/_deploy_penguins_model.joblib
#   03_output/_deploy_penguins_scaler.joblib (when applicable)
#   03_output/__deploy_penguins_new_observations.csv
#   03_output/__deploy_penguins_predictions.csv

# goal:
#   select a preliminary model configuration based on previous results
#   train and save the final model and its feature transformation
#   simulate deployment by loading the saved components
#   and predicting species for new observations

# NOTE: model selection is based on a limited experimental evaluation
# a more rigorous assessment is left for a later MoP

In [ ]:
# ------------------------------------------------------------
# define input and output paths
# ------------------------------------------------------------

from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "02_code": ROOT = ROOT.parent

INPUT = ROOT / "01_input"
OUTPUT = ROOT / "03_output"

assert INPUT.exists(), f"PTS | missing input directory: {INPUT}"
OUTPUT.mkdir( parents=True, exist_ok=True )

print( "input:", INPUT )
print( "output:", OUTPUT )


### [1] From Model Selection to Deployment

In notebook 01, we explored different prediction contexts and constructed modelling datasets using different feature representations.

In notebook 02, we compared prediction scenarios, learning algorithms and feature scaling methods, evaluating their predictive performance.

In this notebook, the emphasis shifts from **predictive performance to the diversity of operations involved in model deployment**.

In this notebook, rather than necessarily choosing the model with the highest accuracy, we will select a configuration that allows us to explore different aspects of:
- preparing,
- saving,
- loading, and
- using a learned model.

The selected configuration should still provide reasonable predictive performance, based on the experiments conducted in notebook 02. However, this is a preliminary, educational choice, not a rigorously validated model selection.

Model deployment requires more than saving a trained classifier. A predictive model also depends on:
- the feature representation used during learning,
- the expected input attributes, including their names, types and order,
- the transformations applied to those attributes, including any parameters learned during training.

Therefore, the deployed system must preserve the classifier and all the information needed to reproduce the **feature representation used during training** when processing new observations.

When new observations are received, their attributes must be validated and arranged according to the model's input requirements. Categorical encoding and numerical scaling must then reproduce the transformations established during training.

**No new learning takes place during prediction:**
- saved transformations are reused through `transform()`, and
- the trained classifier produces predictions through `predict()`.




### [2] Prediction Scenarios and Deployment Requirements

The five prediction scenarios differ in the information required to classify new observations and in the transformations needed to prepare that information for a trained classifier.

| Scenario | Prediction context | Required input attributes | Feature preparation |
|:---|:---|:---|:---|
| **A — historical field study** | Original field-study setting | `island`, `year`, `sex` + 4 morphological measurements | OHE for `island` and `sex`; optional scaling |
| **B — anywhere in Antarctica** | Independent of observation location | `year`, `sex` + 4 morphological measurements | OHE for `sex`; optional scaling |
| **C — whenever needed** | Independent of observation year | `island`, `sex` + 4 morphological measurements | OHE for `island` and `sex`; optional scaling |
| **D — biological profile** | Independent of geographic and temporal context | `sex` + 4 morphological measurements | OHE for `sex`; optional scaling |
| **E — morphological measurements** | Measurements only | 4 morphological measurements | Optional scaling |

The four morphological measurements are `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm` and `body_mass_g`.

#### Processing new observations

The deployed system will receive a CSV file containing one or more observations. Its columns may appear in a different order from those used during training.

The prediction process must:

1. Load the CSV and validate the required attributes.
2. Restore the expected attribute order, independently of the order in the received file.
3. Apply the categorical encoding and numerical scaling required by the selected configuration.
4. Use the saved classifier to predict `species`.
5. Save the predictions to a new CSV file.

Additional columns may be ignored. Required attributes and missing values are checked before prediction. Further validation of data types, invalid values and unknown categories is outside the scope of this MoP.

#### Components to preserve

| Component | Purpose |
|:---|:---|
| Input specification | Required attributes, types and expected order |
| Categorical encoder | Encoding rules and learned categories, when applicable |
| Numerical scaler | Learned scaling parameters, when applicable |
| Feature specification | Names and order of features after transformation |
| Trained classifier | Model used to predict `species` |

**NOTE:** Not every configuration requires all these components. Their necessity depends on the scenario, learning algorithm and transformations selected.


In [ ]:
# ------------------------------------------------------------
# select a configuration for deployment - questions
# ------------------------------------------------------------

# use the results obtained in notebook 02, considering that:
# - predictive performance is relevant, but not the only criterion

# and, also consider this selection CONSTRAINT:
# - EXCLUDE configurations that achieved 100% test accuracy in notebook 02

# this is an educational constraint, not evidence of overfitting or poor generalization
# because such conclusions require further evaluation, including:
# - repeated holdout, or
# - cross-validation to assess performance variability

# the objective is to select a configuration with reasonable predictive performance
# - while preserving the possibility of classification errors

# the selected configuration should also allow us to explore different
# - feature transformations and deployment operations


# question 1: considering the five scenarios, three classifiers
# and three scaling methods evaluated in notebook 02,
# which configuration would you select for deployment?
# justify your choice based on predictive performance and
# the deployment operations you intend to explore

# question 2: which operations will be required to process new
# observations from a CSV file, including attribute ordering,
# feature transformations and model prediction?

# NOTE: this is a preliminary, educational model selection
# a single train/test holdout does not establish the best configuration

In [ ]:
# ------------------------------------------------------------
# define and validate the selected configuration
# ------------------------------------------------------------

# define the selected configuration
# TODO | define the configuration selected and justified above
# REMINDER: select only configurations evaluated in notebook 02
# - including the classifier hyperparameters
# - and, exclude configurations with 100% test accuracy
SCENARIO = ...
CLASSIFIER = ...
CLASSIFIER_PARAMETERS = ...
SCALING = ...


# available configurations from notebook 02
SCENARIOS = ( "A", "B", "C", "D", "E" )
CLASSIFIERS = ( "Tree", "kNN", "Naive Bayes" )
SCALING_METHODS = ( "original", "min-max", "standard" )

# validate the selected configuration
assert SCENARIO in SCENARIOS, "PTS | invalid scenario"
assert CLASSIFIER in CLASSIFIERS, "PTS | invalid classifier"
assert SCALING in SCALING_METHODS, "PTS | invalid scaling method"

assert isinstance( CLASSIFIER_PARAMETERS, dict ), "PTS | classifier parameters must be a dictionary"

# confirm the selected configuration
print( "scenario:", SCENARIO )
print( "classifier:", CLASSIFIER )
print( "parameters:", CLASSIFIER_PARAMETERS )
print( "scaling:", SCALING )

In [ ]:
# ------------------------------------------------------------
# load the selected modelling dataset and its transformer
# ------------------------------------------------------------

import pandas as pd
import joblib

# modelling datasets generated in notebook 01
scenario_files = {
  "A": "penguins_modelling_A_study.csv",
  "B": "penguins_modelling_B_anywhere.csv",
  "C": "penguins_modelling_C_whenever.csv",
  "D": "penguins_modelling_D_biological.csv",
  "E": "penguins_modelling_E_morphological.csv"
}

# load the dataset corresponding to the selected scenario
DATA_PATH = OUTPUT / scenario_files[ SCENARIO ]
data = pd.read_csv( DATA_PATH )

# load the fitted feature transformer from notebook 01
TRANSFORMER_PATH = Path( DATA_PATH ).with_suffix( ".joblib" )
# TODO | load the saved feature transformer
transformer = joblib.load( ...

# recover original input attributes and expected feature representation
input_attributes = transformer.feature_names_in_.tolist()
output_features = transformer.get_feature_names_out().tolist()

# separate features, target and observation identifier
# TODO | extract modelling features and target values
X = ...
y = ...

# preserve the feature names and their order
feature_names = X.columns.tolist()
assert feature_names == output_features, "PTS | modelling features do not match saved transformer"

print( "scenario:", SCENARIO )
print( "dataset:", DATA_PATH.name )
print( "observations:", len( data ) )
print()
print( "transformer:", TRANSFORMER_PATH.name )
print( f"input-attributes:\t | {len( input_attributes )} | {input_attributes}" )
print( f"output-features:\t | {len( output_features )} | {output_features}" )
print( "target:", y.name )


# expected output (partial):
# observations: 333
# target: species
# input-attributes and output-features depend on the selected scenario
# and recall that neither includes the observation identifier (ID) or target (species)

In [ ]:
# ------------------------------------------------------------
# train the selected classifier
# ------------------------------------------------------------

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# select the classifier
classifier_types = {
  "Tree": DecisionTreeClassifier,
  "kNN": KNeighborsClassifier,
  "Naive Bayes": GaussianNB
}

# select the scaling method
scaler_types = {
  "original": None,
  "min-max": MinMaxScaler(),
  "standard": StandardScaler()
}

# get classifier and scaler
# TODO | instantiate the selected classifier using its parameters
classifier = classifier_types[ ... ]( **... )
# TODO | select the scaling method specified in the configuration
scaler = scaler_types[ ... ]

# learn and apply feature scaling
X_train = X.copy()

if scaler is not None:
  scaler.fit( X_train )
  X_train_scaled = scaler.transform( X_train )
  X_train = pd.DataFrame( data=X_train_scaled, columns=X.columns, index=X.index )

# train the selected classifier
# TODO | train the classifier using all available modelling observations
classifier.fit( ..., ... )

print( "scenario:\t", SCENARIO )
print( "classifier:\t", CLASSIFIER )
print( "scaling:\t", SCALING )
print( "observations:\t", X_train.shape[ 0 ] )
print( "features:\t", X_train.shape[ 1 ] )

# expected output (partial):
# observations: 333
# features: depends on the selected scenario

In [ ]:
# ------------------------------------------------------------
# save the trained model and scaler
# ------------------------------------------------------------

import joblib

MODEL_PATH = OUTPUT / "_deploy_penguins_model.joblib"
SCALER_PATH = OUTPUT / "_deploy_penguins_scaler.joblib"

# save the trained classifier
joblib.dump( classifier, MODEL_PATH )

# save the fitted scaler, when applicable
if scaler is not None: joblib.dump( scaler, SCALER_PATH )

print( "saved model:", MODEL_PATH )
print( "saved scaler:", SCALER_PATH if scaler is not None else "not required" )

In [ ]:
# ------------------------------------------------------------
# load the saved components for deployment
# ------------------------------------------------------------

# simulate a new deployment session by:
# - loading the saved components
# - without repeating any learning operations (fit)

import joblib

# load the feature transformer and the trained classifier
deploy_transformer = joblib.load( TRANSFORMER_PATH )
# TODO | load the trained classifier saved for deployment
deploy_classifier = joblib.load( ... )

# load the scaler, when applicable
deploy_scaler = None
if SCALING != "original":
  deploy_scaler = joblib.load( SCALER_PATH )

# recover the expected input attributes and output features
deploy_attributes = deploy_transformer.feature_names_in_.tolist()
deploy_features = deploy_transformer.get_feature_names_out().tolist()

print( "classifier:", type( deploy_classifier ).__name__ )
print( "scaler:", type( deploy_scaler ).__name__ if deploy_scaler is not None else "not required" )
print( "input attributes:", deploy_attributes )
print( "output features:", deploy_features )

In [ ]:
# ------------------------------------------------------------
# create new observations for deployment
# ------------------------------------------------------------

# example observations for the selected scenario
data_new = pd.DataFrame( {
  "body_mass_g":       [ 3800, 4700 ],
  "sex":               [ "FEMALE", "MALE" ],
  "flipper_length_mm": [ 195, 210 ],
  "bill_depth_mm":     [ 17.5, 16.2 ],
  "bill_length_mm":    [ 42.0, 47.5 ],
  "year":              [ 2008, 2009 ],
  "island":            [ "Dream", "Biscoe" ]
} )

# save observations with columns in a different order
PATH_DATA_NEW = OUTPUT / "__deploy_penguins_new_observations.csv"

data_new.to_csv( PATH_DATA_NEW, index=False )

print( "new observations:", PATH_DATA_NEW )
print( "observations:", len( data_new ) )
print( "available attributes:", data_new.columns.tolist() )

In [ ]:
# ------------------------------------------------------------
# load and validate new observations
# ------------------------------------------------------------

# NOTE: input validation is limited to required attributes and missing values
# additional type, category and domain checks are outside the scope of this MoP

# load the CSV received for deployment
data_new = pd.read_csv( PATH_DATA_NEW )

# recover the required attributes from the saved transformer
# TODO | recover the expected input attributes from the saved transformer
attr_required = deploy_transformer. ...

# validate required attributes
attr_missing = [
  attribute for attribute in attr_required
  if attribute not in data_new.columns
]
assert not attr_missing, f"PTS | missing input attributes: {attr_missing}"

# select and order the required attributes
# TODO | select the required attributes in the order expected by the transformer
X_new_raw = data_new[ ... ].copy()

# validate missing values
assert not X_new_raw.isna().any().any(), \
  "PTS | missing values in new observations"

print( "observations:", len( X_new_raw ) )
print( "received attributes:", data_new.columns.tolist() )
print( "required attributes:", attr_required )
print( "deployment input shape:", X_new_raw.shape )

# expected output (partial):
# observations: 2
# deployment input shape: (2, number of required attributes)
# for scenario B: (2, 6)

In [ ]:
# ------------------------------------------------------------
# transform new observations for prediction
# ------------------------------------------------------------

# apply the saved feature representation (including OHE)
# TODO | apply the saved feature transformer to the new observations
values = deploy_transformer.transform( ... )
X_new = pd.DataFrame( data=values, columns=deploy_features, index=X_new_raw.index )

# apply the saved scaling parameters, when applicable
if deploy_scaler is not None:
  # TODO | apply the saved scaler to the transformed features
  values = deploy_scaler.transform( ... )
  X_new = pd.DataFrame( data=values, columns=deploy_features, index=X_new_raw.index )

print( "input shape:", X_new_raw.shape )
print( "transformed shape:", X_new.shape )
print( "features:", X_new.columns.tolist() )


# expected output (partial):
# input shape: (2, number of required attributes)
# transformed shape: (2, number of model features)
# for scenario B: input (2, 6), transformed (2, 7)
# features: depends on the selected scenario

In [ ]:
# ------------------------------------------------------------
# predict species for new observations
# ------------------------------------------------------------

# predict using the saved classifier
# TODO | predict species using the transformed features
y_pred_new = deploy_classifier.predict( ... )

# associate predictions with the received observations
predictions = data_new.copy()
# TODO | add the predicted species to the original observations
predictions[ "predicted_species" ] = ...

# save predictions to a CSV file
PREDICTIONS_PATH = OUTPUT / "__deploy_penguins_predictions.csv"
predictions.to_csv( PREDICTIONS_PATH, index=False )

print( "predictions:", PREDICTIONS_PATH )
print( "observations:", len( predictions ) )

display( predictions )

# expected output (partial):
# observations: 2
# the resulting table contains all original attributes
# plus the column "predicted_species"
# the values of "predicted_species" may depend on the selected configuration

In [ ]:
# ------------------------------------------------------------
# final deployment checkpoint
# ------------------------------------------------------------

# verify that every new observation received a prediction
assert len( predictions ) == len( data_new ), \
  "PTS | prediction count does not match input observations"
assert predictions[ "predicted_species" ].notna().all(), \
  "PTS | missing predictions"

print( "scenario:", SCENARIO )
print( "classifier:", CLASSIFIER )
print( "scaling:", SCALING )
print( "observations processed:", len( predictions ) )
print( "predictions saved:", PREDICTIONS_PATH.name )

# NOTE: deployment reuses saved transformations and a trained classifier
# no fit() operation is performed when processing new observations

In [ ]:
# ------------------------------------------------------------
# explore FunctionTransformer and the identity transformation
# ------------------------------------------------------------

# NOTE (motivation for this additional exercise):
# in the previous deployment example, the "original" scaling option
# required special handling because no scaler was defined (None)
#
# FunctionTransformer provides a common fit()/transform() interface
# for custom transformations, including the identity transformation
#
# using an identity transformer instead of None allows all scaling
# configurations to be processed uniformly, without conditional code

import pandas as pd
from sklearn.preprocessing import FunctionTransformer

# FunctionTransformer wraps a function using the standard
# scikit-learn fit() and transform() interface

# NOTE:
# - FunctionTransformer wraps a function in the standard transformer interface
# - without a function, it implements the identity: f(X) = X
# - the identity can also be expressed as lambda X: X
# - both transformations use the same fit()/transform() operations
#
# NOTE (persistence):
# - lambda functions are not supported by standard joblib persistence
# - ... so, lambda-based transformers are illustrative here
# - reusable custom functions should be defined in an importable module
#
# also notice that:
# - FunctionTransformer() implements the identity transformation by default
# - and supports joblib persistence without requiring a custom function
# ... so identity may be properly implememted using FunctionTransformer()


# example dataset
X_example = pd.DataFrame( {
  "x1": [ 1.0, 2.0 ],
  "x2": [ 3.0, 4.0 ]
} )

# define three alternative transformations
# TODO | consult the FunctionTransformer documentation to understand
# the func parameter and complete the two identity transformations
transformers = {
  "double": FunctionTransformer( func=lambda X: X * 2 ),

  # TODO | define the identity transformation using a lambda function
  "identity_with_lambda": FunctionTransformer( func=... ),

  # TODO | define the persistent identity transformation
  "identity_persistent": FunctionTransformer( func=... )
}

# apply all three transformations using the same operations
for name, transformer in transformers.items():
  transformer.fit( X_example )
  values = transformer.transform( X_example )

  print( f"\n{name}:" )
  print( values )

    # expected output (partial):
# double: [2.0, 6.0] and [4.0, 8.0]
# identity_with_lambda: [1.0, 3.0] and [2.0, 4.0]
# identity_persistent: [1.0, 3.0] and [2.0, 4.0]